# Qwen3-4B — SC·SI × R1·R2·R3 핵심 비교

**8B에서 관찰한 풀이에 따른 억제 절 효과가 4B에서도 나타나는지 확인합니다.** 8B의 전체 실험을 반복하는 것이 아니라 아래 여섯 조건만 비교합니다.

| 피드백 | SC: 공통 재검토 문장 | SI: 공통 + 억제 절 |
|---|---:|---:|
| R1: 실제 정답 선택지만 | 새240개 | 새240개 |
| R2: 같은 정답 + 해당 문제 풀이 | 새240개 | 새240개 |
| R3: 같은 정답 + 다른 과목 풀이 | 새240개 | 새240개 |

4B가 처음 틀린167문항 중 기존 풀이가 있는 **80문항 × 6조건 × 3표현 = 새1,440개**, 배치16으로 **90배치**입니다. 4B 자신의 최초 답변은 그대로 재사용합니다.

기존4B S0/S1 결과와8B 결과는 참고표·모델 비교에 재사용합니다. **4B의 SC/SI는 아직 없어 여섯 셀 모두 새로 생성합니다.** 기존 결과를 다른 조건으로 바꿔 쓰지 않습니다.

**A100 선택 → 1번 설치 → 세션 다시 시작 → 2~8번 순서대로 실행. 설정은 그대로 둡니다.**

자료는 `sycophancy_qwen_4b_replication1440_inputs.zip` 하나를 2번 셀에서 올립니다. 이전 결과 ZIP은 추가로 올리지 않습니다.

두 모델이 공통으로 처음 틀린 **61문항**은 같은 풀이를 사용하며, 별도 모델 비교표를 만듭니다. 풀이 토큰 수 차이5% 이내인76문항의 보조 분석도 저장합니다. 사람이 풀이를 새로 검수할 필요는 없습니다.


## 1. 설치

이 셀만 먼저 실행합니다. 끝나면 **런타임 → 세션 다시 시작** 후2번부터 실행합니다.


In [ ]:
import os, sys, subprocess
INSTALL_RAN_IN_THIS_KERNEL = True
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                   "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
packages = ["transformers==4.57.6", "accelerate==1.12.0", "bitsandbytes==0.48.2",
            "huggingface-hub==0.36.0", "tokenizers==0.22.2", "numpy==2.2.6", "pandas==2.2.3",
            "pyarrow==20.0.0", "matplotlib==3.10.3", "scipy==1.15.3", "tqdm==4.67.1"]
print("1/3 패키지 설치 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("2/3 Transformers 설치 파일 확인·복구 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                       "--no-deps", "--no-cache-dir", "transformers==4.57.6"])
print("3/3 별도 Python 프로세스에서 실제 Qwen3 import 검사", flush=True)
PROBE = """
from transformers.generation import GenerationMixin
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from transformers import BitsAndBytesConfig
import torch, accelerate, bitsandbytes, transformers
print("Qwen3 / GenerationMixin / 양자화 import OK")
print("torch:", torch.__version__, "transformers:", transformers.__version__)
"""
result = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("설치/import 검사 실패. 이후 셀로 넘어가지 말고 출력을 공유하세요.")
print("설치 및 import 점검 완료. 지금 세션을 다시 시작하고 2번부터 실행하세요.", flush=True)


## 2. Drive 연결·자료 ZIP

Drive 연결을 허용하고 이번 `inputs.zip` 하나를 선택합니다. 직접 압축을 풀지 않습니다. 다음 접속부터는 Drive에서 자동으로 불러옵니다.


In [ ]:
import os, sys, hashlib, io, zipfile, json, importlib
from pathlib import Path
if globals().get("INSTALL_RAN_IN_THIS_KERNEL", False):
    raise RuntimeError("런타임 → 세션 다시 시작을 누른 뒤 2번부터 실행하세요.")
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                    "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
from google.colab import drive, files
drive.mount("/content/drive")
INPUT_ZIP_NAME = "sycophancy_qwen_4b_replication1440_inputs.zip"
INPUT_SHA256 = "dd2a0912a15ae22c75009a51a8e4a46527de56fd39e7b62b22d55340067a1042"
CACHE_DIR = Path("/content/drive/MyDrive/sycophancy_qwen/inputs") / INPUT_SHA256[:12]
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = CACHE_DIR / INPUT_ZIP_NAME
if ZIP_PATH.exists():
    raw = ZIP_PATH.read_bytes()
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive의 자료 ZIP이 손상됐습니다. 이 오류 출력을 공유하세요.")
    print("Drive에 보관된 자료 ZIP을 불러왔습니다. 다시 업로드할 필요 없습니다.")
else:
    print("아래 파일 선택 버튼에서", INPUT_ZIP_NAME, "하나를 선택하세요.", flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("자료 ZIP 하나만 선택하고 2번을 다시 실행하세요.")
    raw = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("지정한 자료 ZIP과 내용이 다릅니다. 이번 inputs.zip을 선택하고 2번을 다시 실행하세요.")
    temp = ZIP_PATH.with_suffix(".zip.tmp")
    temp.write_bytes(raw)
    if hashlib.sha256(temp.read_bytes()).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive 저장 검사 실패. 저장 공간과 연결을 확인하고 2번을 다시 실행하세요.")
    temp.replace(ZIP_PATH)
    print("자료 ZIP을 Drive에 보관했습니다. 다음부터 자동으로 불러옵니다.")
CODE_DIR = Path("/content") / ("qwen4b_replication1440_" + INPUT_SHA256[:12])
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if any(not (CODE_DIR / name).resolve().is_relative_to(CODE_DIR.resolve()) for name in z.namelist()):
        raise RuntimeError("잘못된 ZIP 내부 경로")
    z.extractall(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
for name in ["protocol", "engine", "analysis", "runtime_check", "progress_ui",
             "replication_protocol", "replication_run", "replication_analysis", "prepare_inputs"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from runtime_check import check_runtime
RUNTIME = check_runtime()
from replication_protocol import load_source, default_config, freeze, DEFENSES
from replication_run import ReplicationBackend, bind_backend, run_followups, scan_saved, export_results, write_status
from progress_ui import activity
from protocol import read_json
import pandas as pd
from IPython.display import display, Image
print("자료 준비 완료. 3번부터 실행하세요.")


## 3. 원본 확인·설정 고정

**80문항 / 새1,440개 / 90배치 / 공통61문항**을 확인하세요. 기존 최초 답변과 결과, 풀이 배정에 문제가 있으면 중단합니다.


In [ ]:
from replication_protocol import load_history, load_matching, DEFENSE_LABELS
with activity("4B 최초 답변·기존4B/8B 결과·풀이 배정 검사"):
    SOURCE = load_source(CODE_DIR / "source")
    HISTORY8 = load_history(SOURCE)
    MATCHING = load_matching()
CONFIG = default_config(SOURCE)
STUDY_ROOT = Path("/content/drive/MyDrive/sycophancy_qwen/sycophancy_qwen3_4b_replication1440_a100_b16_v1")
with activity("1,440개 새 작업과 분석 계획 고정"):
    FROZEN, JOBS = freeze(STUDY_ROOT, SOURCE, CONFIG)
print("4B 초기 오답80문항 | 새 후속 응답1,440개 | 90배치 | 두 모델 공통 초기 오답61문항")
print("4B 최초300개: 정답125 / 오답167 / 무효8. 오답167개 중 기존 풀이가 있는80개를 사용합니다.")
print("최초 답변 새 생성0개. 기존4B S0/S1 및8B 결과 재생성0개.")
print("4B SC/SI는 아직 없으므로 여섯 셀을 모두 새로 생성합니다. S0/S1과 SC/SI는 서로 다른 조건입니다.")
display(pd.DataFrame({"조건":["R1: 실제 정답 선택지만", "R2: 정답 + 해당 문제 풀이", "R3: 정답 + 다른 과목 풀이"],
                      "SC: 공통 문장":["새240개"]*3, "SI: 공통+억제 절":["새240개"]*3}))
print("8B와 공통인61문항은 같은 R2/R3 풀이를 씁니다. 각 모델의 최초 답변은 그대로 유지합니다.")
print("방어 문구 원문:")
for name, text in DEFENSES.items():
    print(name, ":", text)
print("풀이 길이 차이5% 이내:", MATCHING["summary"]["within_5_percent"], "/80문항. 전체80개 주 분석 + 76개 보조 분석.")
TOKEN_AUDIT = read_json(CODE_DIR / "source/token_audit.json")
print("사전 입력 길이:", TOKEN_AUDIT["max_input_tokens_observed"], "/4096토큰. 4번에서 모든 입력을 다시 대조합니다.")
print("A100 / Qwen3-4B / NF4·BF16 / 비사고 / 배치16 / 출력 최대192토큰. 설정 변경 불필요.")
print("Drive 저장:", STUDY_ROOT)
print("동결 해시:", FROZEN["study_hash"])


## 4. 모델 준비·입력 검사

Qwen3-4B를 다운로드합니다. A100·핵심 생성 환경을 원본과 대조하고1,440개 입력 길이를 확인합니다. 자동으로 입력을 자르거나 배치를 줄이지 않습니다.


In [ ]:
with activity("저장 배치 확인"):
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if len(SAVED) == 90:
    BACKEND = None
    print("이미 1,440개가 저장돼 모델 로딩을 건너뜁니다. 7번에서 분석할 수 있습니다.")
else:
    if globals().get("BACKEND") is None:
        with activity("Qwen3-4B 가중치 다운로드·A100 적재"):
            BACKEND = ReplicationBackend(SOURCE, CONFIG)
    bind_backend(STUDY_ROOT, FROZEN, SOURCE, BACKEND, JOBS)
    print("GPU:", BACKEND.environment["gpu"])
    print("원본과 모델/생성 설정 일치. NF4 / BF16 / 비사고 / 배치 16 / 최대 192토큰")
    if BACKEND.source_environment_notes:
        print("기록한 기타 환경 차이:", BACKEND.source_environment_notes)
    print("모델 및 전체 입력 점검 완료. 5번으로 진행하세요.")


## 5. 첫 배치 점검

첫16개 저장과 출력 형식을 확인합니다. 이16개도 본 실험에 포함되며 다시 생성하지 않습니다.


In [ ]:
SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if 0 not in SAVED:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND, max_new_batches=1)
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
else:
    print("첫 배치는 이미 저장되어 재사용합니다.")
from replication_analysis import score_records
FIRST_CHECK = score_records(SOURCE, SAVED[0]["records"])
print("첫 배치 Drive 저장·해시 검사 통과: 16/16개")
print("형식 실패:", int(FIRST_CHECK.invalid.sum()), "/16 | 잘림:", int(FIRST_CHECK.truncated.sum()), "/16")
print("현재 전체 저장:", sum(len(x["records"]) for x in SAVED.values()), "/1440")
if FIRST_CHECK.invalid.all():
    raise RuntimeError("첫 배치 16개가 모두 형식 실패입니다. 저장 자료는 보존하고 이 출력을 공유하세요.")
print("6번을 실행하면 저장된 배치를 건너뛰고 이어갑니다.")


## 6. 나머지 생성

**1,440/1,440개, 90/90배치, complete=true**를 확인하세요. 진행률·저장 수·남은 시간과 긴 배치의 대기 상태가 표시됩니다.


In [ ]:
if BACKEND is None:
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
    if len(SAVED) != 90:
        raise RuntimeError("4번 모델 준비부터 다시 실행하세요.")
    STATUS = write_status(STUDY_ROOT, SAVED, JOBS, CONFIG, "complete")
    print("이미 모든 배치가 저장되어 재생성하지 않습니다.")
else:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND)
    if STATUS["complete"]:
        BACKEND.close()
        BACKEND = None
print(json.dumps(STATUS, ensure_ascii=False, indent=2))


## 7. 자동 분석

80문항의 여섯 조건과 공통61문항 모델 비교를 계산합니다. 유리한 결과만 고르지 않고 효과 크기·구간·형식 실패를 함께 봅니다.


In [ ]:
from replication_analysis import analyze
with activity("4B80문항 자동 채점·공통61문항 모델 비교·그래프"):
    REPORT_DIR, TABLE, EFFECTS, ASSESSMENT = analyze(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS)
print("4B 정답 수정률(%): 같은80문항, 각 셀240개 응답")
display(TABLE.pivot(index="condition", columns="defense", values="final_correct_percent").reindex(["R1","R2","R3"]).reindex(columns=["SC","SI"]).round(2))
print("R1=정답만 / R2=해당 문제 풀이 / R3=다른 과목 풀이. SC=공통 문장 / SI=공통+억제 절.")
display(TABLE.rename(columns={"condition":"피드백", "defense":"방어", "n_items":"문항수", "n_responses":"응답수",
    "correct_count":"정답 수정 수", "final_correct_percent":"정답 수정률", "initial_wrong_retained_percent":"초기 오답 유지율",
    "other_wrong_change_percent":"다른 오답", "invalid_percent":"형식 실패", "truncated_percent":"잘림"}).round(2))
VIEW = ["label_ko","n_items","difference_pp","ci_low_pp","ci_high_pp"]
print("핵심 비교: (R2의 SI−SC)−(R3의 SI−SC). 문항 단위95% 구간.")
display(EFFECTS[EFFECTS.analysis_role=="primary"][VIEW].round(2))
print("SI 자체의 수정률도 확인: R2−R3 (보조 비교)")
display(EFFECTS[EFFECTS.contrast.isin(["SI_related_minus_swapped","SC_related_minus_swapped"])][VIEW].round(2))
for message in ASSESSMENT["messages_ko"]:
    print(message)
print("두 모델이 공통으로 처음 틀린61문항의 수정률(%), 각 셀183개 응답")
COMMON = pd.read_csv(REPORT_DIR / "common61_comparison.csv")
display(COMMON.pivot(index="condition",columns=["model","defense"],values="final_correct_percent").round(2))
print("4B80문항과8B77문항의 비율만 비교하면 대상 문항 차이가 섞입니다. 모델 비교는 위 공통61문항을 먼저 보세요.")
print("길이 차이5% 이내인76문항 보조 분석")
display(pd.read_csv(REPORT_DIR / "length_sensitivity_effects.csv")[VIEW].round(2))
print("형식 실패 보조 진단: 엄격한 주 채점은 바꾸지 않습니다.")
display(pd.read_csv(REPORT_DIR / "format_diagnostic_counts.csv"))
for name in ["01_4b_comparison.png","02_4b_effects.png","03_common61_models.png"]:
    display(Image(filename=str(REPORT_DIR/name)))
print("저장:", REPORT_DIR)


## 8. 결과 ZIP 다운로드

원응답·설정·분석표·그래프가 담긴 ZIP을 다운로드합니다. Drive에도 저장됩니다. 중단 중에도 저장된 자료를 내보낼 수 있습니다.


In [ ]:
with activity("결과 ZIP을 Drive에 저장"):
    RESULT_ZIP = export_results(STUDY_ROOT)
print("완료 상태:", read_json(STUDY_ROOT / "status.json") if (STUDY_ROOT / "status.json").exists() else "미실행")
files.download(str(RESULT_ZIP))


## 저장·재개

저장: `내 드라이브/sycophancy_qwen/sycophancy_qwen3_4b_replication1440_a100_b16_v1/`

배치마다 Drive에 저장합니다. 새 A100 세션에서는 **1번 설치 → 세션 다시 시작 → 2~8번** 순서로 재개합니다. 저장된 배치를 확인해 건너뛰므로 처음부터 다시 생성하지 않습니다. 같은 폴더로 동시에 여러 노트북을 실행하지 마세요.

이번 결과는 초기 오답에 정답을 제시했을 때의 수정 행동을 다룹니다. 최초 정답 유지·오답 권위 유도·전체A실험을 반복하지 않습니다. 8B와 동일한 핵심 방법을 쓰지만 모델별 최초 답변과 실행 순서는 다릅니다.

제작 시 기존에 캐시한 공식4B 토크나이저로 실제 입력 길이를 확인하고, 가상 응답으로 저장·재개·통계·내보내기를 검증했습니다. 실제 A100 추론은 이 노트북에서 실행합니다.
